# SECTION 1 — PROJECT OVERVIEW

## # US Electric Car Registrations & Renewable Energy Production  


## SkillCircle – Data Analytics Capstone Project

## 1. Project Overview

Electric vehicles (EVs) and renewable energy are two major pillars of a sustainable future.  
This project analyzes EV registrations across US states and compares them with renewable energy production trends.

## 1.1 Objective

- Import and clean multiple datasets  
- Perform exploratory data analysis (EDA)  
- Visualize EV and energy trends  
- Analyze correlations  
- Conduct time series analysis  
- Compare states using geospatial-style charts  
- Build a simple predictive model  
- Provide insights and recommendations

## 1.2 Datasets Used

1. **States_Electric_Vehicle_Registrations_2018.xlsx**  
2. **States_All_Vehicle_Registrations_2018.xlsx**  
3. **States_Annual_Energy_Generation_Sources_1990_2019.xlsx**  
4. **state_codes.xlsx**

---


## SECTION 2 — IMPORT LIBRARIES & LOAD DATA

In [2]:

import pandas as pd

In [3]:
import numpy as np

In [4]:
import matplotlib.pyplot as plt

In [5]:
import seaborn as sns

In [6]:
pd.set_option('display.max_columns', 100)

In [7]:
sns.set(style="whitegrid", palette="viridis")

In [8]:
plt.rcParams['figure.figsize'] = (12, 6)

## SECTION 3 — LOAD ALL DATASETS

### 3.1 Loading datasets.

In [50]:
ev_2018 = pd.read_excel("States_Electric_Vehicle_Registrations_2018.xlsx")

In [51]:
all_vehicles = pd.read_excel("States_All_Vehicle_Registrations_2018.xlsx")

In [ ]:
energy = pd.read_excel("States_Annual_Energy_Generation_Sources_1990_2019.xlsx")

In [53]:
state_codes = pd.read_excel("state_codes.xlsx")

In [54]:
ev_2018.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,Electric Vehicle Registrations by State,NaN,NaN,NaN
1,NaN,State,Registration Count,NaN,NaN
2,NaN,Alabama,1450,NaN,NaN
3,NaN,Alaska,530,NaN,NaN
4,NaN,Arizona,15000,NaN,NaN


## SECTION 4 — DATA CLEANING & PREPROCESSING

In [55]:
# Clean column names
ev_2018.columns = ev_2018.columns.str.strip().str.lower().str.replace(" ", "_")

In [ ]:
all_vehicles.columns = all_vehicles.columns.str.strip().str.lower().str.replace(" ", "_")

In [57]:
energy.columns = energy.columns.str.strip().str.lower().str.replace(" ", "_")

In [58]:
state_codes.columns = state_codes.columns.str.strip().str.lower().str.replace(" ", "_")

In [59]:
# Rename columns for consistency
ev_2018.rename(columns={"state": "state_name", "registration_count": "ev_registrations"}, inplace=True)

In [60]:
all_vehicles.rename(columns={"state": "state_name"}, inplace=True)

In [61]:
energy.rename(columns={"year": "year", "state": "state_code"}, inplace=True)

In [ ]:
print(ev_2018.columns)
print(state_codes.columns)
print(all_vehicles.columns)

Index(['unnamed:_0', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4'], dtype='object')
Index(['state_code', 'state_name'], dtype='object')
Index(['', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4',
       'unnamed:_5', 'unnamed:_6', 'unnamed:_7', 'unnamed:_8', 'unnamed:_9',
       'unnamed:_10', 'unnamed:_11', 'unnamed:_12', 'unnamed:_13',
       'unnamed:_14', 'unnamed:_15'],
      dtype='object')


In [64]:
ev_2018 = pd.read_excel("States_Electric_Vehicle_Registrations_2018.xlsx", skiprows=2)

In [ ]:
# Merge EV data with state codes
ev_merged = ev_2018.merge(state_codes, how="Left", on="state_name")

In [ ]:
# Merge EV + All Vehicles
ev_vehicle_merged = ev_merged.merge(all_vehicles, how="left", on="state_name")

In [ ]:
ev_vehicle_merged.head()

## SECTION 4 — DATA CLEANING & PREPROCESSING

In [73]:
# Clean column names
ev_2018.columns = ev_2018.columns.str.strip().str.lower().str.replace(" ", "_")
all_vehicles.columns = all_vehicles.columns.str.strip().str.lower().str.replace(" ", "_")
energy.columns = energy.columns.str.strip().str.lower().str.replace(" ", "_")
state_codes.columns = state_codes.columns.str.strip().str.lower().str.replace(" ", "_")

In [74]:
# Rename columns for consistency
ev_2018.rename(columns={"state": "state_name", "registration_count": "ev_registrations"}, inplace=True)
all_vehicles.rename(columns={"state": "state_name"}, inplace=True)
energy.rename(columns={"year": "year", "state": "state_code"}, inplace=True)


In [75]:
# Merge EV data with state codes
ev_merged = ev_2018.merge(state_codes, how="left", on="state_name")


In [ ]:
# Merge EV + All Vehicles
ev_vehicle_merged = ev_merged.merge(all_vehicles, how="left", on="state_name")

In [ ]:
ev_vehicle_merged.head()

## SECTION 5 — EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# Summary statistics
ev_vehicle_merged.describe().T

## EV Registrations Distribution

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(ev_vehicle_merged["ev_registrations"], bins=20, kde=True)
plt.title("Distribution of EV Registrations (2018)")
plt.xlabel("EV Registrations")
plt.ylabel("Frequency")
plt.show()


## Top 10 States by EV Registrations

In [ ]:
top10 = ev_vehicle_merged.sort_values("ev_registrations", ascending=False).head(10)

sns.barplot(data=top10, x="ev_registrations", y="state_name")
plt.title("Top 10 States by EV Registrations (2018)")
plt.xlabel("EV Registrations")
plt.ylabel("State")
plt.show()


## SECTION 6 — ENERGY DATA PROCESSING

In [ ]:
# Filter only renewable sources
renewable_sources = ["Hydroelectric Conventional", "Wind", "Solar Thermal and Photovoltaic", "Wood and Wood Derived Fuels"]

energy_renewable = energy[energy["energy_source"].isin(renewable_sources)]

In [ ]:
# Group by state and year
energy_state_year = energy_renewable.groupby(["state_code", "year"])["generation_(megawatthours)"].sum().reset_index()
energy_state_year.rename(columns={"generation_(megawatthours)": "renewable_generation"}, inplace=True)


In [ ]:
energy_state_year.head()

## SECTION 7 — MERGE EV + ENERGY DATA

In [ ]:
# Merge EV + Energy using state_code
final_df = ev_merged.merge(energy_state_year, how="left", on="state_code")

final_df.head()


## SECTION 8 — CORRELATION ANALYSIS

In [ ]:
corr_df = final_df[["ev_registrations", "renewable_generation"]].dropna()

sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation: EV Registrations vs Renewable Energy Generation")
plt.show()


## SECTION 9 — TIME SERIES ANALYSIS

In [ ]:
energy_ts = energy_state_year.groupby("year")["renewable_generation"].sum().reset_index()

plt.plot(energy_ts["year"], energy_ts["renewable_generation"], marker="o")
plt.title("US Renewable Energy Generation Over Time (1990–2019)")
plt.xlabel("Year")
plt.ylabel("Renewable Generation (MWh)")
plt.show()


## SECTION 10 — STATE-WISE COMPARISON (GEOSPATIAL STYLE)

In [ ]:
state_compare = final_df.groupby("state_name")[["ev_registrations", "renewable_generation"]].sum().reset_index()

sns.scatterplot(data=state_compare, x="renewable_generation", y="ev_registrations")
plt.title("State-wise Comparison: EV Registrations vs Renewable Energy")
plt.xlabel("Renewable Energy Generation")
plt.ylabel("EV Registrations")
plt.show()


## SECTION 11 — SIMPLE PREDICTIVE MODEL

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

model_df = final_df.dropna(subset=["ev_registrations", "renewable_generation"])

X = model_df[["renewable_generation"]]
y = model_df["ev_registrations"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


## SECTION 12 — CONCLUSION & RECOMMENDATIONS

### Conclusion & Recommendations

#### Key Findings
- California, Florida, and Georgia lead in EV registrations.
- Renewable energy production has steadily increased from 1990 to 2019.
- States with higher renewable energy generation tend to show higher EV adoption.
- Correlation suggests a positive relationship between clean energy and EV growth.

### Recommendations
- Increase renewable energy investments to support EV adoption.
- Expand charging infrastructure in low-EV states.
- Provide tax incentives and subsidies for EV buyers.
- Encourage public-private partnerships for clean mobility.

#### Final Note
This project demonstrates the relationship between EV adoption and renewable energy production using real US datasets.  
It includes data cleaning, EDA, correlation, time series analysis, modeling, and insights — suitable for LMS submission, LinkedIn, and resume.
